In [ ]:
import json

with open("<TODO>/basque_linguistic_competence/logs/data-all.json", "r", encoding="utf-8") as f:
    data = json.loads(f.read())

del f

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, integrate

GENERAL_BENCHMARKS = [
    "arc_eu_challenge_mc", "arc_eu_easy_mc", "belebele_eus_Latn", "bertaqa_eu_global",
    "bertaqa_eu_local", "eus_exams_eu", "eus_reading", "eus_trivia", "mmlu_eu", "piqa_eu_mc",
]

types, x, y = [], [], []
for key, value in data.items():
    if "_60M-2-new" in key or "LOGPROBS" not in value:
        continue
    lp, bm = value["LOGPROBS"], value["BENCHMARK"]
    is_base = "bat-ctxt" in lp
    types.append("Base" if is_base else "Instruct")
    y.append(np.mean(lp["bat-ctxt" if is_base else "bat-inst"]) * 100)
    x.append(np.mean([bm[b] for b in GENERAL_BENCHMARKS]) * 100)
types, x, y = np.array(types), np.array(x), np.array(y)
N = len(x)

In [ ]:
def fisher_ci(r, n, conf=0.95):
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    q = stats.norm.ppf(0.5 + conf / 2)
    return np.tanh(z - q * se), np.tanh(z + q * se)

def tost_corr(r, n, delta):
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    p_lower = stats.norm.sf((z - np.arctanh(-delta)) / se)
    p_upper = stats.norm.cdf((z - np.arctanh(delta)) / se)
    return max(p_lower, p_upper)

def bf01_pearson(r, n):
    f = lambda rho: (1 - rho**2) ** ((n - 1) / 2) * (1 - rho * r) ** (-(2 * n - 3) / 2)
    return 1 / (0.5 * integrate.quad(f, -1, 1)[0])

r, p = stats.pearsonr(x, y)
lo, hi = fisher_ci(r, N)
print(f"n = {N}, r = {r:.3f}, 95% CI [{lo:.2f}, {hi:.2f}], p = {p:.2f}, BF01 = {bf01_pearson(r, N):.1f}, TOST (|rho| >= 0.4) p = {tost_corr(r, N, 0.4):.3f}")

In [ ]:
sns.set_style("whitegrid")
plt.rcParams.update({
    "font.family": "serif", "font.size": 14, "axes.labelsize": 16, "xtick.labelsize": 14,
    "ytick.labelsize": 14, "figure.dpi": 150, "savefig.dpi": 300, "grid.linestyle": "--", "grid.alpha": 0.6,
})
STYLE = {"Base": dict(c="#191970", marker="o"), "Instruct": dict(c="white", marker="s")}

fig, ax = plt.subplots(figsize=(6, 6))
for kind in ("Base", "Instruct"):
    mask = types == kind
    ax.scatter(y[mask], x[mask], s=100, edgecolor="black", linewidth=1.5, zorder=3,
               label=f"{kind} (n = {mask.sum()})", **STYLE[kind])
ax.set_xlim(50, 76); ax.set_ylim(48, 85)
ax.set_xticks(range(50, 76, 5)); ax.set_yticks(range(50, 86, 5))
ax.set_xlabel("BAT Accuracy Score (IPM)", weight="bold")
ax.set_ylabel("General-Purpose Benchmark Average Score", weight="bold")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False, fontsize=16, markerscale=1.2,
          handletextpad=0.2, columnspacing=1.0)
sns.despine(trim=True, offset=5)
plt.tight_layout(pad=0)
plt.savefig("BAT-benchmarks-stats.pdf", bbox_inches="tight", pad_inches=0)
plt.show()